# Tutorial 4: Molecular Descriptors with ASE（分子記述子とASEデータベース）

**所要時間**: 30-40分

**学習内容**:
- ASEデータベースの作成と管理
- 分子記述子の計算（分子量、官能基、π共役比）
- SMARTSパターンマッチング
- カスタムデータセットでの学習
- 記述子による厳密な条件付き生成

**注意**: このチュートリアルでは一切の近似的なフォールバック処理を行いません。全ての計算は厳密に実行されます。

In [ ]:
# セットアップ
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from ase import Atoms
from ase.db import connect
import os

# 分子記述子抽出関数をインポート
from qm9.openbabel_functions import (
    extract_molecular_descriptors_ase_openbabel,
    extract_atom_types_from_ase,
    extract_molecular_weight_from_ase,
    extract_functional_groups_openbabel,
    extract_pi_conjugation_ratio_openbabel,
    get_functional_group_patterns,
    is_openbabel_available
)

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'OpenBabel available: {is_openbabel_available()}')
print('\n=== Tutorial 4: 分子記述子とASEデータベース ===')
print('フォールバック処理なし - 全て厳密計算')

## 1. ASEデータベースの作成

ASE (Atomic Simulation Environment) データベースは、分子構造と関連するプロパティを保存するための標準的なフォーマットです。

### 1.1 基本的なデータベース作成

In [ ]:
# データベースファイルを作成（既存の場合は上書き）
db_path = 'tutorial_descriptors.db'
if os.path.exists(db_path):
    os.remove(db_path)
    
db = connect(db_path, append=False)

# サンプル分子を定義（厳密な原子座標を使用）
molecules_data = [
    {
        'name': 'Ethane',
        'symbols': ['C', 'C', 'H', 'H', 'H', 'H', 'H', 'H'],
        'positions': [
            [0.000, 0.000, 0.000],   # C1
            [1.540, 0.000, 0.000],   # C2
            [-0.513, 0.889, 0.000],  # H1
            [-0.513, -0.444, 0.889], # H2
            [-0.513, -0.444, -0.889],# H3
            [2.053, 0.889, 0.000],   # H4
            [2.053, -0.444, 0.889],  # H5
            [2.053, -0.444, -0.889], # H6
        ],
        'properties': {'target_mw': 30.07}  # 期待される分子量
    },
    {
        'name': 'Methanol',
        'symbols': ['C', 'O', 'H', 'H', 'H', 'H'],
        'positions': [
            [0.000, 0.000, 0.000],   # C
            [1.430, 0.000, 0.000],   # O
            [-0.513, 0.889, 0.000],  # H1
            [-0.513, -0.444, 0.889], # H2
            [-0.513, -0.444, -0.889],# H3
            [1.750, 0.000, 0.950],   # H (OH)
        ],
        'properties': {'target_mw': 32.04}
    },
    {
        'name': 'Water',
        'symbols': ['O', 'H', 'H'],
        'positions': [
            [0.000, 0.000, 0.000],   # O
            [0.757, 0.586, 0.000],   # H1
            [-0.757, 0.586, 0.000],  # H2
        ],
        'properties': {'target_mw': 18.015}
    },
]

# データベースに分子を追加
for mol_data in molecules_data:
    atoms = Atoms(
        symbols=mol_data['symbols'],
        positions=mol_data['positions']
    )
    
    # プロパティを保存
    db.write(atoms, name=mol_data['name'], data=mol_data['properties'])
    
print(f'✓ Created ASE database: {db_path}')
print(f'✓ Added {len(molecules_data)} molecules')
print(f'✓ Database contains {db.count()} entries')

## 2. 分子記述子の抽出

分子記述子は分子の化学的特性を数値で表現したものです。

### 2.1 基本記述子（ASEのみで計算可能）

In [ ]:
# データベースから分子を読み込んで記述子を計算
print('\n=== 基本記述子の計算 ===\n')

for i, row in enumerate(db.select(), 1):
    atoms = row.toatoms()
    mol_name = row.get('name', f'Molecule_{i}')
    
    print(f'{i}. {mol_name}')
    print(f'   化学式: {atoms.get_chemical_formula()}')
    print(f'   原子数: {len(atoms)}')
    
    # 原子タイプの抽出（厳密計算）
    atom_types = extract_atom_types_from_ase(atoms)
    print(f'   原子タイプ: {atom_types}')
    
    # 分子量の計算（厳密計算）
    molecular_weight = extract_molecular_weight_from_ase(atoms)
    print(f'   計算分子量: {molecular_weight:.3f} u')
    
    # 期待値との比較（厳密性の検証）
    if 'target_mw' in row.data:
        expected_mw = row.data['target_mw']
        error = abs(molecular_weight - expected_mw)
        print(f'   期待分子量: {expected_mw:.3f} u')
        print(f'   誤差: {error:.3f} u ({error/expected_mw*100:.2f}%)')
        
        # 誤差が大きい場合は警告（フォールバックなし）
        if error > 0.5:
            print(f'   ⚠️  WARNING: Large deviation detected!')
    
    print()

### 2.2 高度な記述子（OpenBabel使用）

OpenBabelを使用すると、官能基やπ共役比などの高度な記述子を計算できます。

In [ ]:
# OpenBabelが利用可能な場合のみ実行
if is_openbabel_available():
    print('\n=== 高度な記述子の計算（OpenBabel） ===\n')
    
    for i, row in enumerate(db.select(), 1):
        atoms = row.toatoms()
        mol_name = row.get('name', f'Molecule_{i}')
        
        # 全記述子を一度に抽出
        descriptors = extract_molecular_descriptors_ase_openbabel(atoms)
        
        print(f'{i}. {mol_name}')
        print(f'   原子タイプ: {descriptors["atom_types"]}')
        print(f'   分子量: {descriptors["molecular_weight"]:.3f} u')
        print(f'   官能基: {descriptors["functional_groups"] if descriptors["functional_groups"] else "なし"}')
        print(f'   π共役比: {descriptors["pi_conjugation_ratio"]:.3f}')
        print()
else:
    print('\n⚠️  OpenBabel not available. Install with: pip install openbabel-wheel')
    print('高度な記述子計算にはOpenBabelが必要です。')

## 3. SMARTSパターンマッチング

SMARTS (SMILES Arbitrary Target Specification) は、分子の部分構造を検索するための言語です。

### 3.1 利用可能な官能基パターン

In [ ]:
# 利用可能な官能基パターンのリストを表示
if is_openbabel_available():
    print('\n=== 利用可能な官能基パターン ===\n')
    
    patterns = get_functional_group_patterns()
    print(f'合計 {len(patterns)} 種類の官能基パターンが定義されています:\n')
    
    for i, pattern in enumerate(patterns, 1):
        print(f'{i:2d}. {pattern}')
    
    print('\n各パターンの説明:')
    descriptions = {
        'hydroxyl': 'ヒドロキシル基 (-OH)',
        'carbonyl': 'カルボニル基 (C=O)',
        'carboxyl': 'カルボキシル基 (-COOH)',
        'aldehyde': 'アルデヒド基 (-CHO)',
        'ketone': 'ケトン基',
        'amino': 'アミノ基 (-NH2, -NH-)',
        'nitro': 'ニトロ基 (-NO2)',
        'chloro': 'クロロ基 (-Cl)',
        'bromo': 'ブロモ基 (-Br)',
        'fluoro': 'フルオロ基 (-F)',
        'iodo': 'ヨード基 (-I)',
        'methyl': 'メチル基 (-CH3)',
        'methoxy': 'メトキシ基 (-OCH3)',
        'phenyl': 'フェニル基（ベンゼン環）',
    }
    
    for pattern in patterns:
        if pattern in descriptions:
            print(f'  • {pattern}: {descriptions[pattern]}')
else:
    print('SMARTSパターンマッチングにはOpenBabelが必要です。')

### 3.2 カスタムデータセットの作成

より複雑な分子を含むカスタムデータセットを作成します。

In [ ]:
# より複雑な分子を含むカスタムデータベースを作成
custom_db_path = 'tutorial_custom_molecules.db'
if os.path.exists(custom_db_path):
    os.remove(custom_db_path)
    
custom_db = connect(custom_db_path, append=False)

# 様々な官能基を持つ分子を追加
complex_molecules = [
    {
        'name': 'Formaldehyde',  # アルデヒド
        'symbols': ['C', 'O', 'H', 'H'],
        'positions': [
            [0.000, 0.000, 0.000],
            [1.210, 0.000, 0.000],
            [-0.580, 0.942, 0.000],
            [-0.580, -0.942, 0.000],
        ],
    },
    {
        'name': 'Ammonia',  # アミノ基的
        'symbols': ['N', 'H', 'H', 'H'],
        'positions': [
            [0.000, 0.000, 0.000],
            [0.960, 0.000, 0.000],
            [-0.480, 0.831, 0.000],
            [-0.480, -0.415, 0.719],
        ],
    },
]

for mol_data in complex_molecules:
    atoms = Atoms(
        symbols=mol_data['symbols'],
        positions=mol_data['positions']
    )
    custom_db.write(atoms, name=mol_data['name'])

print(f'✓ Created custom database: {custom_db_path}')
print(f'✓ Added {len(complex_molecules)} molecules')

# 記述子を計算して表示
if is_openbabel_available():
    print('\n=== カスタム分子の記述子 ===\n')
    
    for i, row in enumerate(custom_db.select(), 1):
        atoms = row.toatoms()
        mol_name = row.get('name', f'Molecule_{i}')
        
        descriptors = extract_molecular_descriptors_ase_openbabel(atoms)
        
        print(f'{i}. {mol_name}')
        print(f'   化学式: {atoms.get_chemical_formula()}')
        print(f'   分子量: {descriptors["molecular_weight"]:.3f} u')
        print(f'   官能基: {descriptors["functional_groups"]}')
        print()

## 4. カスタムデータセットでの学習

ASEデータベースを使用したモデルの学習方法を説明します。

### 4.1 学習コマンド例

In [ ]:
# 学習コマンドの生成
print('\n=== カスタムデータセットでの学習コマンド ===\n')

train_command = f'''python main_qm9.py \\\n    --dataset ase_db \\\n    --ase_db_path {db_path} \\\n    --conditioning molecular_weight \\\n    --exp_name tutorial_ase_model \\\n    --n_epochs 100 \\\n    --batch_size 32 \\\n    --lr 1e-4
'''

print('基本的な学習:')
print(train_command)

print('\n複数の記述子を使用した学習 (OpenBabel必須):')
train_command_advanced = f'''python main_qm9.py \\\n    --dataset ase_db \\\n    --ase_db_path {db_path} \\\n    --conditioning molecular_weight pi_conjugation_ratio \\\n    --exp_name tutorial_ase_advanced \\\n    --n_epochs 100 \\\n    --batch_size 32
'''
print(train_command_advanced)

print('\n注意事項:')
print('  • フォールバック処理は一切行われません')
print('  • 記述子の計算が失敗した場合、明示的なエラーが発生します')
print('  • 全ての条件付けは厳密に実行されます')

## 5. 厳密な条件付き生成

学習したモデルを使用して、特定の分子記述子を持つ分子を生成します。

### 5.1 単一プロパティでの生成

In [ ]:
print('\n=== 厳密な条件付き生成 ===\n')

# 単一プロパティでの生成
generation_command_1 = '''python eval_conditional_qm9.py \\\n    --model_path outputs/tutorial_ase_model \\\n    --n_samples 100 \\\n    --conditioning molecular_weight \\\n    --property_values "molecular_weight=50.0" \\\n    --use_exact_conditions True
'''

print('1. 単一プロパティ（分子量）での生成:')
print(generation_command_1)

print('\n特徴:')
print('  • molecular_weight=50.0 を厳密に満たす分子を生成')
print('  • 近似的な条件付けは行わない')
print('  • 条件を満たせない場合はエラーを返す')

# 複数プロパティでの生成
generation_command_2 = '''python eval_conditional_qm9.py \\\n    --model_path outputs/tutorial_ase_advanced \\\n    --n_samples 100 \\\n    --conditioning molecular_weight pi_conjugation_ratio \\\n    --property_values "molecular_weight=50.0,pi_conjugation_ratio=0.3" \\\n    --use_exact_conditions True
'''

print('\n2. 複数プロパティでの生成:')
print(generation_command_2)

print('\n検証方法:')
print('  • 生成された分子の記述子を再計算')
print('  • 指定した条件との誤差を厳密に評価')
print('  • 許容誤差を超える場合は警告を出力')

### 5.2 生成結果の検証（シミュレーション）

In [ ]:
# 生成結果の検証をシミュレート
print('\n=== 生成結果の検証例 ===\n')

# シミュレーションデータ
target_mw = 50.0
tolerance = 1.0  # 許容誤差 (u)

# 生成された分子の例（実際の生成結果をシミュレート）
generated_results = [
    {'id': 1, 'mw': 50.2, 'formula': 'C3H6O'},
    {'id': 2, 'mw': 49.8, 'formula': 'C3H6O'},
    {'id': 3, 'mw': 50.1, 'formula': 'C4H2'},
    {'id': 4, 'mw': 51.5, 'formula': 'C3H5N'},  # 許容範囲外
]

print(f'目標分子量: {target_mw} u')
print(f'許容誤差: ±{tolerance} u')
print(f'許容範囲: [{target_mw-tolerance}, {target_mw+tolerance}] u\n')

valid_count = 0
for result in generated_results:
    error = abs(result['mw'] - target_mw)
    is_valid = error <= tolerance
    
    status = '✓' if is_valid else '✗'
    if is_valid:
        valid_count += 1
    
    print(f'{status} 分子 {result["id"]}: '
          f'{result["formula"]:8s} '
          f'MW={result["mw"]:5.1f}u '
          f'誤差={error:4.1f}u')
    
    if not is_valid:
        print(f'  → 許容範囲外！ フォールバック処理なしのため要再生成')

print(f'\n成功率: {valid_count}/{len(generated_results)} ({valid_count/len(generated_results)*100:.1f}%)')
print('\n重要: 条件を満たさない分子は自動的に修正されません。')
print('      厳密性を保つため、条件を満たす分子のみが有効とされます。')

## 6. 完全なワークフロー例

実際の研究で使用できる完全なワークフローを示します。

In [ ]:
print('\n=== 完全なワークフロー ===\n')

workflow = '''
【ステップ1: データ準備】
1. ASEデータベースを作成
2. 分子構造と座標を追加
3. 記述子を事前計算（オプション）

【ステップ2: 記述子計算】
4. 各分子の記述子を厳密に計算
5. 計算結果の検証（期待値との比較）
6. 異常値の検出と排除

【ステップ3: モデル学習】
7. 記述子を条件付けとして使用
8. 拡散モデルの学習
9. 学習曲線の監視

【ステップ4: 条件付き生成】
10. 目標記述子を指定
11. 分子を生成
12. 生成結果の記述子を再計算

【ステップ5: 検証】
13. 目標値との誤差を計算
14. 許容範囲内の分子のみを採用
15. 構造の妥当性を確認

【重要な原則】
• 全ステップで厳密な計算を実施
• フォールバック処理は一切行わない
• エラーは明示的に報告
• 数値精度を常に監視
'''

print(workflow)

# ワークフローの実行例
print('\nPythonスクリプト例:')
workflow_script = '''
# complete_workflow.py
import os
from ase.db import connect
from qm9.openbabel_functions import extract_molecular_descriptors_ase_openbabel

# 1. データベース作成
db = connect('my_molecules.db')

# 2. 記述子計算と検証
for row in db.select():
    atoms = row.toatoms()
    
    # 記述子を厳密に計算
    descriptors = extract_molecular_descriptors_ase_openbabel(atoms)
    
    # 検証: 分子量が期待範囲内か
    if descriptors['molecular_weight'] < 10 or descriptors['molecular_weight'] > 500:
        print(f"Warning: Unusual molecular weight {descriptors['molecular_weight']}")
        continue
    
    # 記述子をデータベースに保存
    db.update(row.id, data=descriptors)

print("Workflow completed successfully")
'''
print(workflow_script)

## まとめ

### 学習した内容

✅ **ASEデータベースの作成と管理**
   - 分子構造の保存方法
   - プロパティの関連付け
   - データベースの読み書き

✅ **分子記述子の厳密な計算**
   - 原子タイプの抽出
   - 分子量の計算
   - 官能基の検出（OpenBabel）
   - π共役比の計算（OpenBabel）

✅ **SMARTSパターンマッチング**
   - 14種類の官能基パターン
   - カスタムパターンの定義方法
   - パターンマッチングの実行

✅ **カスタムデータセットでの学習**
   - ASEデータベースを入力として使用
   - 複数の記述子での条件付け
   - 学習パラメータの設定

✅ **厳密な条件付き生成**
   - 目標記述子の指定
   - 生成結果の検証
   - 誤差の評価

### 重要な原則

1. **フォールバックなし**: 全ての計算は厳密に実行される
2. **明示的エラー**: 失敗時は明確なエラーメッセージ
3. **数値精度**: 常に計算精度を監視
4. **検証必須**: 生成結果は必ず検証する

### 次のステップ

次のチュートリアルでは、生成された分子の包括的な評価と解析方法を学びます：
- 安定性メトリクス
- RDKit検証
- 一意性・新規性の評価
- プロパティ分布の解析

**Next: Tutorial 5 - 評価と解析**